# Proyecto 1 — Limpieza y preparación de datos

Etapas iniciales (10 % + 30 %): cargar y diagnosticar el archivo **Características y composición del hogar**. Todavía no se limpian datos ni se genera un archivo procesado. El CSV original permanece intacto en `datos/raw/`.

## 1. Librerías

Usamos `pandas` para leer la tabla y `Path` para ubicar el archivo dentro del proyecto.

In [1]:
import pandas as pd
from pathlib import Path

## 2. Ruta del archivo original

La ruta funciona si el notebook se ejecuta desde la raíz del proyecto o desde la carpeta `notebooks`.

In [2]:
ruta_proyecto = Path.cwd()
if ruta_proyecto.name == 'notebooks':
    ruta_proyecto = ruta_proyecto.parent

ruta_datos = ruta_proyecto / 'datos' / 'raw' / 'caracteristicas_composicion_hogar.csv'
print('Archivo de entrada:', ruta_datos)
print('¿Existe el archivo?', ruta_datos.exists())

Archivo de entrada: C:\Users\c.collazost\Desktop\E. Bases y algoritmos\repo_transfer_aplicaciones_proyecto1\datos\raw\caracteristicas_composicion_hogar.csv
¿Existe el archivo? True


## 3. Carga de los datos

El archivo separa columnas con punto y coma (`;`) y escribe decimales con coma. Por eso se indican `sep=';'` y `decimal=','`.

In [3]:
df = pd.read_csv(ruta_datos, sep=';', decimal=',')
print('Filas y columnas:', df.shape)

Filas y columnas: (235350, 78)


## 4. Inspección inicial

Revisamos nombres de columnas, algunas filas y tipos de datos. La vista de filas se limita a 12 columnas para que sea fácil de leer. Aún no interpretamos los códigos de las variables.

In [4]:
print('Nombres de las columnas:')
print(df.columns.tolist())

display(df.iloc[:3, :12])
df.info()

Nombres de las columnas:
['DIRECTORIO', 'SECUENCIA_ENCUESTA', 'SECUENCIA_P', 'ORDEN', 'FEX_C', 'P6016', 'P1894', 'P6020', 'P6034', 'P6040', 'P6051', 'P5502', 'P6071', 'P6071S1', 'P756', 'P756S1', 'P756S2', 'P756S3', 'P3510', 'P3510S1', 'P3510S1A1', 'P3510S1A2', 'P3510S1A3', 'P3510S1A4', 'P3510S1A5', 'P3510S1A6', 'P3510S1A7', 'P3510S1A8', 'P3510S1A9', 'P3510S1A10', 'P3510S1A11', 'P3510S2', 'P3510S2A1', 'P3510S2A2', 'P3510S2A3', 'P3510S2A4', 'P3510S2A5', 'P3510S2A6', 'P3510S2A7', 'P3510S2A8', 'P3510S2A9', 'P3510S2A10', 'P3510S2A11', 'P6074', 'P755', 'P755S1', 'P755S2', 'P755S3', 'P754', 'P753', 'P753S1', 'P753S2', 'P753S3', 'P752', 'P1662', 'P6081', 'P6081S1', 'P6087', 'P6083', 'P6083S1', 'P6088', 'P6080', 'P2057', 'P2059', 'P2061', 'P1895', 'P1896', 'P1897', 'P1898', 'P1899', 'P3175', 'P1901', 'P1903', 'P1904', 'P1905', 'P1927', 'P3038', 'P3039']


,DIRECTORIO,SECUENCIA_ENCUESTA,SECUENCIA_P,ORDEN,FEX_C,P6016,P1894,P6020,P6034,P6040,P6051,P5502
0,8369550,1,1,1,472.131901,1,3,2,1,45,1,5.0
1,8369550,2,1,2,472.131901,2,3,1,1,25,7,4.0
2,8369550,3,1,3,472.131901,3,2,2,1,16,9,5.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 235350 entries, 0 to 235349
Data columns (total 78 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   DIRECTORIO          235350 non-null  int64  
 1   SECUENCIA_ENCUESTA  235350 non-null  int64  
 2   SECUENCIA_P         235350 non-null  int64  
 3   ORDEN               235350 non-null  int64  
 4   FEX_C               235350 non-null  float64
 5   P6016               235350 non-null  int64  
 6   P1894               235350 non-null  int64  
 7   P6020               235350 non-null  int64  
 8   P6034               235350 non-null  int64  
 9   P6040               235350 non-null  int64  
 10  P6051               235350 non-null  int64  
 11  P5502               202953 non-null  float64
 12  P6071               96746 non-null   float64
 13  P6071S1             92899 non-null   float64
 14  P756                235350 non-null  int64  
 15  P756S1              79013 non-null

## 5. Unidad de análisis

El [diccionario oficial del DANE para la ECV 2025](https://microdatos.dane.gov.co/index.php/catalog/905/data-dictionary/F9?file_name=Caracteristicas+y+composicion+del+hogar) describe este archivo como registros de las personas que conforman los hogares. Coincide con nuestras 235.350 filas y 78 variables. `DIRECTORIO` puede repetirse porque varias personas pertenecen al mismo directorio; eso no es por sí solo un error.

In [5]:
print('Registros de personas:', len(df))
print('Directorios diferentes:', df['DIRECTORIO'].nunique())
print('Filas con DIRECTORIO ya visto:', df.duplicated(subset=['DIRECTORIO']).sum())

Registros de personas: 235350
Directorios diferentes: 86848
Filas con DIRECTORIO ya visto: 148502


## 6. Revisión de duplicados

Un duplicado exacto repite toda la fila. También comprobamos si se repiten combinaciones de columnas de identificación. La combinación `DIRECTORIO` + `SECUENCIA_P` + `ORDEN` es una **clave candidata observada en este archivo**, no una clave oficial confirmada por el diccionario. No eliminamos nada en esta etapa.

In [6]:
columnas_registro = ['DIRECTORIO', 'SECUENCIA_P', 'ORDEN']

print('Filas exactamente duplicadas:', df.duplicated().sum())
print('DIRECTORIO + ORDEN repetidos:', df.duplicated(subset=['DIRECTORIO', 'ORDEN']).sum())
print('Clave candidata repetida:', df.duplicated(subset=columnas_registro).sum())

Filas exactamente duplicadas: 0
DIRECTORIO + ORDEN repetidos: 454
Clave candidata repetida: 0


## 7. Valores faltantes, tipos y valores únicos

`isna().sum()` cuenta vacíos por columna; `isna().mean() * 100` calcula su porcentaje. `nunique()` cuenta valores diferentes. Esta tabla es un diagnóstico: una pregunta que no aplica a una persona también puede quedar vacía, así que no se deben rellenar o borrar todos los faltantes automáticamente.

In [7]:
resumen_variables = pd.DataFrame({
    'tipo': df.dtypes,
    'faltantes': df.isna().sum(),
    'porcentaje_faltantes': (df.isna().mean() * 100).round(2),
    'valores_unicos': df.nunique()
})

print('Columnas con al menos un valor faltante:', (resumen_variables['faltantes'] > 0).sum())
print('Columnas completamente vacías:', (resumen_variables['faltantes'] == len(df)).sum())
print('Nombres de esas columnas:', resumen_variables[resumen_variables['faltantes'] == len(df)].index.tolist())
display(resumen_variables.sort_values('faltantes', ascending=False).head(20))

Columnas con al menos un valor faltante: 63
Columnas completamente vacías: 2
Nombres de esas columnas: ['P3510S1', 'P3510S2']


,tipo,faltantes,porcentaje_faltantes,valores_unicos
P3510S2,float64,235350,100.00,0
P3510S1,float64,235350,100.00,0
P3510S1A7,float64,235349,100.00,1
P3510S2A10,float64,235349,100.00,1
P3510S2A7,float64,235347,100.00,1
P3510S1A9,float64,235345,100.00,1
P3510S1A8,float64,235345,100.00,1
P3510S1A10,float64,235345,100.00,1
P3510S2A5,float64,235344,100.00,1
P3510S1A5,float64,235343,100.00,1


## 8. Comprobación de columnas clave

Miramos por separado los identificadores y tres variables cuyo significado aparece en el diccionario del DANE: `FEX_C` es el factor de expansión, `P6020` pregunta por sexo al nacer y `P6040` por edad cumplida. Que pandas lea un código como número no significa que debamos calcular su promedio.

In [8]:
columnas_clave = ['DIRECTORIO', 'SECUENCIA_ENCUESTA', 'SECUENCIA_P', 'ORDEN',
                  'FEX_C', 'P6020', 'P6040']
display(resumen_variables.loc[columnas_clave])

,tipo,faltantes,porcentaje_faltantes,valores_unicos
DIRECTORIO,int64,0,0.0,86848
SECUENCIA_ENCUESTA,int64,0,0.0,21
SECUENCIA_P,int64,0,0.0,6
ORDEN,int64,0,0.0,21
FEX_C,float64,0,0.0,57219
P6020,int64,0,0.0,2
P6040,int64,0,0.0,107


## Siguiente etapa

Antes de modificar la base, revisaremos con el cuestionario qué vacíos corresponden a preguntas condicionadas y si hay valores imposibles o códigos especiales. `DIRECTORIO` repetido no se tratará como duplicado. `FEX_C` se conservará como factor de expansión. Esta etapa solo diagnostica; no altera el CSV original ni crea todavía una copia procesada.